# LeakAudit - 03 - Protocol collapse on D1 (E4) - core result

**Hold the model, data and features fixed; vary only the evaluation protocol.**

- P1 naive random -> P2 sensor-independent -> P3 site-independent -> P4 site + temporal.
- Three tasks: 5-class, leak vs non-leak, and outdoor vs indoor (the responsibility-attribution decision).
- Results are appended to `results/E4_collapse.csv` and are resumable.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # make 'src' importable from notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation
cfg.apply_mpl_style()
import time
X1, X1_max, M1, HZ = io_utils.load_d1()
y5   = M1.cls.to_numpy()
site = M1.site.to_numpy(); sid = M1.sid.to_numpy()
dt   = pd.to_datetime(M1.dt).to_numpy()

OUT = cfg.RESULTS / "E4_collapse.csv"
SUBSAMPLE = 30000   # cap on training rows, for tractable run time

## Task and protocol grid

In [2]:
TASKS = {
    "T5_five_class":      (np.ones(len(y5), bool), y5),
    "TL_leak_vs_nonleak": (np.ones(len(y5), bool),
                           np.where(np.isin(y5, ["out", "in"]), "leak", "nonleak")),
    "TR_out_vs_in":       (np.isin(y5, ["out", "in"]), y5),
}
MODELS = ["gbdt", "rf", "logreg", "mlp"]   # four model families, to show the
                                           # collapse is not model-specific
SEEDS  = range(cfg.N_SEEDS)

done = set()
if OUT.exists():
    _d = pd.read_csv(OUT)
    done = set(zip(_d.task, _d.protocol, _d.model, _d.seed))
    print(f"resuming: {len(done)} results already on disk")

## Run (resumable)

In [3]:
t0 = time.time()
for tname, (mask, yfull) in TASKS.items():
    Xs, ys = X1[mask], yfull[mask]
    gsite, gsid, gdt = site[mask], sid[mask], dt[mask]
    for kind in MODELS:
        for seed in SEEDS:
            for proto in evaluation.PROTOCOLS:
                if (tname, proto, kind, seed) in done:
                    continue
                sp = evaluation.get_split(proto, ys, gsite, gsid, gdt, seed)
                if sp is None:
                    continue
                a, b = sp
                if len(b) < 30 or len(np.unique(ys[a])) < 2 or len(np.unique(ys[b])) < 2:
                    continue
                if len(a) > SUBSAMPLE:
                    a = np.random.RandomState(seed).choice(a, SUBSAMPLE, replace=False)
                r = evaluation.evaluate(Xs[a], ys[a], Xs[b], ys[b], kind, seed)
                rec = dict(task=tname, protocol=proto, model=kind, seed=seed,
                           n_train=len(a), n_test=len(b),
                           n_tr_site=len(set(gsite[a])), n_te_site=len(set(gsite[b])), **r)
                pd.DataFrame([rec]).to_csv(OUT, mode="a",
                                           header=not OUT.exists(), index=False)
                print(f"  {tname:20s} {proto:18s} {kind:5s} s{seed} "
                      f"acc={r['acc']:.4f} base={r['base']:.4f}", flush=True)
print(f"\nelapsed {time.time() - t0:.0f} s")

  T5_five_class        P1_naive_random    gbdt  s0 acc=0.8930 base=0.3149
  T5_five_class        P2_sensor_indep    gbdt  s0 acc=0.5425 base=0.3126
  T5_five_class        P3_site_indep      gbdt  s0 acc=0.4787 base=0.3879
  T5_five_class        P4_site_temporal   gbdt  s0 acc=0.4202 base=0.2856
  T5_five_class        P1_naive_random    gbdt  s1 acc=0.8927 base=0.3149
  T5_five_class        P2_sensor_indep    gbdt  s1 acc=0.5545 base=0.3407
  T5_five_class        P3_site_indep      gbdt  s1 acc=0.3160 base=0.4620


c:\Users\ecredible\anaconda3\envs\ai-paper\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  T5_five_class        P4_site_temporal   gbdt  s1 acc=0.4433 base=0.2468
  T5_five_class        P1_naive_random    gbdt  s2 acc=0.8947 base=0.3149
  T5_five_class        P2_sensor_indep    gbdt  s2 acc=0.5214 base=0.2968
  T5_five_class        P3_site_indep      gbdt  s2 acc=0.5851 base=0.4614
  T5_five_class        P4_site_temporal   gbdt  s2 acc=0.2781 base=0.3832
  T5_five_class        P1_naive_random    gbdt  s3 acc=0.8939 base=0.3149
  T5_five_class        P2_sensor_indep    gbdt  s3 acc=0.5675 base=0.3353
  T5_five_class        P3_site_indep      gbdt  s3 acc=0.4062 base=0.2980
  T5_five_class        P4_site_temporal   gbdt  s3 acc=0.2640 base=0.3540


c:\Users\ecredible\anaconda3\envs\ai-paper\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  T5_five_class        P1_naive_random    gbdt  s4 acc=0.8915 base=0.3149
  T5_five_class        P2_sensor_indep    gbdt  s4 acc=0.5314 base=0.3159
  T5_five_class        P3_site_indep      gbdt  s4 acc=0.4596 base=0.2841
  T5_five_class        P4_site_temporal   gbdt  s4 acc=0.2377 base=0.2979


...

  T5_five_class        P1_naive_random    mlp   s4 acc=0.8875 base=0.3149
  T5_five_class        P2_sensor_indep    mlp   s4 acc=0.5097 base=0.3159
  T5_five_class        P3_site_indep      mlp   s4 acc=0.4419 base=0.2841
  T5_five_class        P4_site_temporal   mlp   s4 acc=0.2276 base=0.2979


c:\Users\ecredible\anaconda3\envs\ai-paper\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  T5_five_class        P1_naive_random    mlp   s5 acc=0.9035 base=0.3149
  T5_five_class        P2_sensor_indep    mlp   s5 acc=0.5125 base=0.2926
  T5_five_class        P3_site_indep      mlp   s5 acc=0.4868 base=0.5345
  T5_five_class        P4_site_temporal   mlp   s5 acc=0.2700 base=0.3928


c:\Users\ecredible\anaconda3\envs\ai-paper\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  T5_five_class        P1_naive_random    mlp   s6 acc=0.8914 base=0.3149
  T5_five_class        P2_sensor_indep    mlp   s6 acc=0.5111 base=0.3269
  T5_five_class        P3_site_indep      mlp   s6 acc=0.4820 base=0.3912
  T5_five_class        P4_site_temporal   mlp   s6 acc=0.3784 base=0.3222
  T5_five_class        P1_naive_random    mlp   s7 acc=0.8997 base=0.3149
  T5_five_class        P2_sensor_indep    mlp   s7 acc=0.5222 base=0.3194
  T5_five_class        P3_site_indep      mlp   s7 acc=0.5926 base=0.4913
  T5_five_class        P4_site_temporal   mlp   s7 acc=0.2333 base=0.4196
  T5_five_class        P1_naive_random    mlp   s8 acc=0.8898 base=0.3149
  T5_five_class        P2_sensor_indep    mlp   s8 acc=0.4991 base=0.2982
  T5_five_class        P3_site_indep      mlp   s8 acc=0.6006 base=0.5312
  T5_five_class        P4_site_temporal   mlp   s8 acc=0.2063 base=0.3679
  T5_five_class        P1_naive_random    mlp   s9 acc=0.9022 base=0.3149
  T5_five_class        P2_sensor_indep

## Summary

In [4]:
d4 = pd.read_csv(OUT)
# Aggregate PER MODEL. Pooling across model families would average a strong
# learner with weaker ones and report a mixture that describes none of them.
s4 = (d4.groupby(["task", "model", "protocol"])
        .agg(acc=("acc", "mean"), acc_sd=("acc", "std"),
             bacc=("bacc", "mean"), base=("base", "mean"), n=("acc", "size"))
        .reset_index())
s4["above_base"] = s4.acc - s4.base
order = {p: i for i, p in enumerate(evaluation.PROTOCOLS)}
s4 = s4.sort_values(["task", "model", "protocol"],
                    key=lambda c: c.map(order) if c.name == "protocol" else c)
print(s4.round(4).to_string(index=False))
s4.to_csv(cfg.RESULTS / "E4_collapse_summary.csv", index=False)

# The headline model for the paper; the others are the robustness check that
# the collapse is not specific to one learner.
HEADLINE_MODEL = "gbdt"

print(f"\ncollapse (P1 -> worst protocol), model = {HEADLINE_MODEL}:")
for t in s4.task.unique():
    sub = s4[(s4.task == t) & (s4.model == HEADLINE_MODEL)]
    if not len(sub):
        continue
    p1 = sub[sub.protocol == "P1_naive_random"].acc.iloc[0]
    wi = sub.acc.idxmin()
    flag = "  *** below baseline ***" if sub.loc[wi, "acc"] < sub.loc[wi, "base"] else ""
    print(f"  {t:22s} {p1:.4f} -> {sub.loc[wi,'acc']:.4f} "
          f"({sub.loc[wi,'protocol']}), base {sub.loc[wi,'base']:.4f}{flag}")

print("\ncollapse magnitude per model (P1 minus worst):")
for t in s4.task.unique():
    for mdl in s4.model.unique():
        sub = s4[(s4.task == t) & (s4.model == mdl)]
        if not len(sub):
            continue
        p1 = sub[sub.protocol == "P1_naive_random"].acc.iloc[0]
        print(f"  {t:22s} {mdl:7s} {p1:.4f} -> {sub.acc.min():.4f} "
              f"(drop {p1 - sub.acc.min():.4f})")

              task  model         protocol    acc  acc_sd   bacc   base  n  above_base
     T5_five_class   gbdt  P1_naive_random 0.8935  0.0018 0.8572 0.3149 10      0.5786
     T5_five_class   gbdt  P2_sensor_indep 0.5435  0.0168 0.4237 0.3156 10      0.2279
     T5_five_class   gbdt    P3_site_indep 0.4806  0.0949 0.3732 0.4289 10      0.0517
     T5_five_class   gbdt P4_site_temporal 0.3272  0.1104 0.3376 0.3406 10     -0.0134
     T5_five_class logreg  P1_naive_random 0.6277  0.0028 0.5277 0.3149 10      0.3127
     T5_five_class logreg  P2_sensor_indep 0.5132  0.0155 0.4034 0.3156 10      0.1976
     T5_five_class logreg    P3_site_indep 0.4655  0.0979 0.3642 0.4289 10      0.0366
     T5_five_class logreg P4_site_temporal 0.3026  0.0920 0.3229 0.3406 10     -0.0380
     T5_five_class    mlp  P1_naive_random 0.8934  0.0087 0.8666 0.3149 10      0.5784
     T5_five_class    mlp  P2_sensor_indep 0.5137  0.0170 0.4094 0.3156 10      0.1981
     T5_five_class    mlp    P3_site_indep 